# Thực hành SplatAD với PandaSet trên Colab

Chuẩn bị dữ liệu trên **CPU** với `PHASE = "prepare"`. Để chạy hết trên **A100**, chọn GPU A100, đặt `PHASE = "train"`, giữ `TRAIN_STAGE = "all"` và chọn **Runtime → Run all**. Lượt thử 500 bước sẽ kiểm tra checkpoint và render trước khi notebook chạy lượt chính 10.000 bước, xuất đánh giá và đóng gói demo.

**Đưa mã nguồn vào Colab:** tải `SplatAD-project.zip` lên `My Drive/SplatAD/project/`. Notebook tự chép mã nguồn vào `/content`. Nếu bạn đã giải nén hoặc clone repo vào Colab, đặt `PROJECT_DIR` đến thư mục có `pandaset_data.py`.

Mặc định dùng sequence **011**, với năm ca từ dễ đến khó trong `cases.json`. Chỉ dùng pickle từ kho PandaSet được liên kết hoặc từ bản dữ liệu của bạn.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
import shutil
import subprocess
import sys
import zipfile

from google.colab import drive

PHASE = "prepare"  # @param ["prepare", "train"]
TRAIN_STAGE = "all"  # @param ["all", "smoke", "main"]
PROJECT_DIR = Path("/content/SplatAD-project-a100")
DRIVE_ROOT = Path("/content/drive/MyDrive/SplatAD")
DATASETS = DRIVE_ROOT / "datasets"
REVIEW = DRIVE_ROOT / "review"
REVIEWED_ZIPS = REVIEW / "datasets"
RAW_ROOT = Path("/content/raw/pandaset")
REVIEWED_ROOT = Path("/content/reviewed/pandaset")
GPU_ROOT = Path("/content/data/pandaset")
RUNS = DRIVE_ROOT / "runs"
DEMO = DRIVE_ROOT / "demo"
# Để trống cho HTTP Range. Hoặc nhập đường dẫn ZIP đầy đủ trong Drive.
LOCAL_FULL_ZIP = ""
RESUME = False
USE_PROJECT_CASES = False  # Bật khi muốn dùng bộ ca trong gói mã nguồn thay cho bản đã lưu trên Drive.

if PHASE not in {"prepare", "train"}:
    raise ValueError("PHASE phải là prepare hoặc train")
if TRAIN_STAGE not in {"all", "smoke", "main"}:
    raise ValueError("TRAIN_STAGE phải là all, smoke hoặc main")
drive.mount("/content/drive")
for directory in (DATASETS, REVIEW, REVIEWED_ZIPS, RUNS):
    directory.mkdir(parents=True, exist_ok=True)

if not (PROJECT_DIR / "pandaset_data.py").is_file():
    project_zip = DRIVE_ROOT / "project/SplatAD-project.zip"
    if not project_zip.is_file():
        raise FileNotFoundError(f"Tải SplatAD-project.zip vào {project_zip.parent}")
    shutil.unpack_archive(project_zip, PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

from IPython.display import display
from pandaset_data import (ARCHIVE, CAMERAS, PAPER_SEQUENCES, HttpRangeFile,
                           pack_sequence, sample_rgb, sequence_ids, shortlist, stage_sequence)
from pandaset_review import (apply_edits, case_track_ids, load_json, overlay, pack_reviewed,
                             track_table, validate_cases, write_json, write_overlay_clip)
from colab_pipeline import PREFIX, audit_parser, export_lidar, group_runs, pack_demo, render, run, train

@contextmanager
def open_source(block_size=None):
    if LOCAL_FULL_ZIP:
        with zipfile.ZipFile(LOCAL_FULL_ZIP) as archive:
            yield archive
    else:
        with HttpRangeFile(ARCHIVE, block_size=block_size) as remote, zipfile.ZipFile(remote) as archive:
            yield archive

cases_path = REVIEW / "cases.json"
cases = load_json(cases_path if cases_path.exists() and not USE_PROJECT_CASES else PROJECT_DIR / "cases.json")
validate_cases(cases)
sequences = sorted({case["sequence"] for case in cases})

## Task 1 — Lọc sequence và xem ảnh gốc

Ô này quét cuboid của các sequence trong bài báo. Bật `SCAN_MORE` để quét phần còn lại khi muốn chọn cảnh mới. Bảng trả về UUID, lớp và frame ứng viên. Cột camera trong annotation là gợi ý để kiểm tra bằng ảnh. Che khuất được xác nhận khi xem ảnh.

Để đổi ca, sửa `My Drive/SplatAD/review/cases.json` rồi chạy lại ô thiết lập. Để nạp bộ ca đi kèm phiên bản mã nguồn mới, đặt `USE_PROJECT_CASES = True` và chạy Task 2 để lưu lựa chọn lên Drive.

In [ ]:
SCAN_MORE = False
SCAN_PAPER_SEQUENCES = True
if PHASE == "prepare" and (SCAN_MORE or SCAN_PAPER_SEQUENCES):
    with open_source() as archive:
        sequences = PAPER_SEQUENCES if not SCAN_MORE else sequence_ids(archive)
        candidates = shortlist(archive, sequences)
        candidates.to_json(REVIEW / "candidate_tracks.json", orient="records", indent=2)
        for kind in ("walking", "cycling"):
            display(candidates[candidates.case == kind].head(12))
        for case in cases:
            for name, frame in case["keyframes"].items():
                print(case["case"], case["sequence"], case["camera"], name, frame)
                display(sample_rgb(archive, case["sequence"], case["camera"], frame))

## Task 2 — Tải ZIP theo sequence

Notebook tải phần thuộc các sequence đã chọn và đóng gói thành `pandaset_<ID>.zip`. Nếu đã có ZIP này, upload thẳng vào `My Drive/SplatAD/datasets/` và đặt `SCAN_PAPER_SEQUENCES = False` trong Task 1. Với bản PandaSet đầy đủ trên máy, có thể tự giải nén sequence và nén lại, giữ cây `pandaset/<ID>/...`. Nếu ZIP đầy đủ đang ở Drive, đặt `LOCAL_FULL_ZIP` thành đường dẫn tuyệt đối trong `/content/drive`.

In [ ]:
if PHASE == "prepare":
    pending = [sequence for sequence in sequences if not (DATASETS / f"pandaset_{sequence}.zip").is_file()]
    if pending:
        with open_source(block_size=8 * 1024**2) as archive:
            for sequence in pending:
                local_zip = Path("/content") / f"pandaset_{sequence}.zip"
                pack_sequence(archive, sequence, local_zip)
                # Publish to Drive only once the archive is complete.
                partial = (DATASETS / local_zip.name).with_suffix(".part")
                shutil.copyfile(local_zip, partial)
                partial.replace(DATASETS / local_zip.name)
                local_zip.unlink()
    write_json(cases_path, cases)

## Task 3 — Nạp ZIP từ Drive và kiểm tra cảm biến

Ở phase GPU, cell này dùng ZIP **đã duyệt** từ Task 5. Dữ liệu huấn luyện được giải nén vào `/content/data/pandaset/<ID>`. PandaSet parser nhận `/content/data/pandaset` làm `--data`.

In [ ]:
if PHASE == "prepare":
    for sequence in sequences:
        print(stage_sequence(DATASETS / f"pandaset_{sequence}.zip", RAW_ROOT, sequence))
    for case in cases:
        frame = list(case["keyframes"].values())[1]
        print(case["case"], "Pandar64 + cuboid gốc", frame)
        display(overlay(RAW_ROOT / case["sequence"], case["camera"], frame, case_track_ids(case)))
else:
    for sequence in sequences:
        print(stage_sequence(REVIEWED_ZIPS / f"pandaset_{sequence}.zip", GPU_ROOT, sequence))

## Task 4 — Rà soát hoặc bổ sung cuboid

Mở overlay và point cloud của đối tượng để kiểm tra. Nếu cần sửa, tạo `My Drive/SplatAD/review/edits_<ID>.json` dưới dạng một danh sách thao tác. Ví dụ minh họa cấu trúc chỉnh yaw, cần thay `UUID` và số liệu theo bằng chứng thực tế.

```json
[{"frame": 21, "uuid": "UUID", "action": "update", "reason": "Đã đối chiếu LiDAR và ảnh", "values": {"yaw": 0.23}}]
```

`update` nhận các trường hình học, `stationary`, `label` và `uuid`. Với `add`, đặt UUID ở ngoài `values` và điền `label`, `position.x/y/z`, `dimensions.x/y/z`, `yaw` trong `values`. Khi thêm hộp cho một track đã có, dùng đúng UUID. Nếu dữ liệu gốc đã phù hợp, nhật ký sửa sẽ là danh sách rỗng.

In [ ]:
if PHASE == "prepare":
    for sequence in sequences:
        edit_path = REVIEW / f"edits_{sequence}.json"
        edits = load_json(edit_path) if edit_path.exists() else []
        apply_edits(RAW_ROOT / sequence, REVIEWED_ROOT / sequence, edits, REVIEW / f"audit_{sequence}.json")
    for case in cases:
        frame = list(case["keyframes"].values())[1]
        print(case["case"], "overlay sau rà soát", frame)
        display(overlay(REVIEWED_ROOT / case["sequence"], case["camera"], frame, case_track_ids(case)))

## Task 5 — Rà soát tracking và lưu bản đã duyệt

Bảng track hiển thị khoảng trống giữa các hộp, mức dịch chuyển tâm và nhãn. Nếu phát hiện nhầm ID, cập nhật `edits_<ID>.json` và chạy lại Task 4. Để trống hộp khi thiếu bằng chứng. Task 6 xuất pose nội suy để có thể kiểm tra ca che khuất.

Nếu đã chạy Task 4 trong phiên Colab trước, bạn có thể chạy ô thiết lập rồi đến thẳng Task 5. Khi thiếu bản đã duyệt trong phiên mới, ô này dựng lại nó từ ZIP trên Drive và tệp chỉnh sửa đã lưu.

In [ ]:
if PHASE == "prepare":
    if shutil.which("ffmpeg") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
    for sequence in sequences:
        if not (REVIEWED_ROOT / sequence).is_dir():
            original = stage_sequence(DATASETS / f"pandaset_{sequence}.zip", RAW_ROOT, sequence)
            edit_path = REVIEW / f"edits_{sequence}.json"
            edits = load_json(edit_path) if edit_path.exists() else []
            apply_edits(original, REVIEWED_ROOT / sequence, edits, REVIEW / f"audit_{sequence}.json")
    for case in cases:
        scene = REVIEWED_ROOT / case["sequence"]
        for actor_index, uuid in enumerate(case_track_ids(case)):
            table = track_table(scene, uuid)
            if len(table) < 2 or table.stationary.any() or table.label.nunique() != 1:
                raise ValueError(f"Track động không hợp lệ cho {case['case']}: {uuid}")
            suffix = "_secondary" if actor_index else ""
            table.assign(uuid=uuid).to_csv(REVIEW / f"track_{case['case']}{suffix}.csv", index=False)
            display(table[table.frame.between(case["start_frame"], case["end_frame"])])
        write_overlay_clip(scene, case, REVIEW / f"overlay_{case['case']}.mp4")
    for sequence in sequences:
        pack_reviewed(REVIEWED_ROOT / sequence, REVIEWED_ZIPS / f"pandaset_{sequence}.zip")
    print("Đã lưu dữ liệu đã duyệt. Chuyển sang Colab GPU và đặt PHASE = train.")

## Task 6 — Cài và kiểm tra PandaSet parser của Nerfstudio

Môi trường dùng Python 3.10, PyTorch 2.0.1, NumPy 1.26.4 và CUDA 11.8. SplatAD dùng bộ giải mã LiDAR MLP PyTorch có sẵn và rasterizer CUDA `gsplat`. Task 6 kiểm tra GPU, rasterizer và CLI, rồi chạy `PandaSetDataParser` để ghi track mục tiêu và danh sách frame train/test.

Log cài đặt nằm ở `My Drive/SplatAD/runs/setup-all.log` khi chạy cả hai lượt. Nếu lỗi, xem các dòng ngay trước thông báo giai đoạn thất bại.

In [ ]:
if PHASE == "train":
    subprocess.run(["nvidia-smi"], check=True)
    setup_log = RUNS / f"setup-{TRAIN_STAGE}.log"
    print(f"Log cài đặt: {setup_log}", flush=True)
    subprocess.run(["bash", "-o", "pipefail", "-c", 'bash "$1" 2>&1 | tee "$2"',
                    "splatad-setup", str(PROJECT_DIR / "setup_colab.sh"), str(setup_log)], check=True)
    smoke_specs = list(group_runs(cases, GPU_ROOT, RUNS, smoke=True)) if TRAIN_STAGE != "main" else []
    main_specs = list(group_runs(cases, GPU_ROOT, RUNS, smoke=False)) if TRAIN_STAGE != "smoke" else []
    for spec in smoke_specs + main_specs:
        spec_file = Path(spec["path"]) / "spec.json"
        write_json(spec_file, spec)
        run(PREFIX / "bin/python", PROJECT_DIR / "colab_task.py", "audit", spec_file)

## Task 7 — Huấn luyện bằng `ns-train` theo lượt đã chọn

Với `TRAIN_STAGE = "all"`, ô này chạy thử 500 bước ở nửa độ phân giải với một camera và Pandar64, xuất RGB, depth và một scan LiDAR, rồi chạy 10.000 bước. `"smoke"` dừng sau lượt thử. `"main"` chạy lượt chính. Nếu phiên bị ngắt giữa lượt đang chạy, dùng cùng dữ liệu và đường dẫn Drive, đặt `RESUME = True` rồi chạy lại.

In [ ]:
if PHASE == "train":
    for spec in smoke_specs:
        cfg = train(spec, resume=RESUME)
        render(spec, cfg)
        run(PREFIX / "bin/python", PROJECT_DIR / "colab_task.py", "lidar", Path(spec["path"]) / "spec.json", cfg)
    if smoke_specs:
        print("Đã xong lượt chạy thử SplatAD.")
    configs = {spec["sequence"]: train(spec, resume=RESUME) for spec in main_specs}

## Task 8 — Render từ checkpoint trên các frame giữ lại

`ns-render dataset` ghi GT RGB, RGB dựng, depth màu và depth thô `.npy.gz`. Lệnh gọi đánh giá của SplatAD để tính PSNR, SSIM và LPIPS trên các ảnh giữ lại. `evaluation.json` lưu điểm từng ảnh và trung bình theo sequence. API LiDAR ghi scan thật và dự đoán thành cặp `.ply`.

In [ ]:
if PHASE == "train" and TRAIN_STAGE != "smoke":
    from PIL import Image
    for spec in main_specs:
        cfg = configs[spec["sequence"]]
        outputs = render(spec, cfg)
        evaluation = load_json(outputs / "evaluation.json")
        print(f"Sequence {spec['sequence']} · {evaluation['frame_count']} frame giữ lại")
        print("PSNR (dB) ↑ · SSIM ↑ · LPIPS ↓", evaluation["mean"])
        run(PREFIX / "bin/python", PROJECT_DIR / "colab_task.py", "lidar", Path(spec["path"]) / "spec.json", cfg)
        for case in spec["cases"]:
            for role, frame in case["keyframes"].items():
                relative = Path(f"{frame:02d}.png")
                if len(spec["cameras"]) > 1:
                    relative = Path(f"{case['camera']}_camera") / relative
                print(case["case"], role, frame, "gốc / dựng")
                display(*(Image.open(outputs / "test" / folder / relative) for folder in ("gt-rgb", "rgb")))

## Task 9 — Đóng gói demo web

Sau khi xem cặp ảnh, có thể thêm ghi chú quan sát bằng `OBSERVATIONS`. Ghi đúng những gì thấy trong ảnh, ví dụ vị trí nhòe ở một frame cụ thể. Dữ liệu crop mặc định lấy từ hình chiếu của cuboid. Có thể sửa bằng `crop: [x, y, width, height]` theo tỉ lệ 0–1.

Tải `demo.zip` từ `My Drive/SplatAD/`, giải nén và mở thư mục demo qua một máy chủ HTTP cục bộ. Xem lệnh trong README.

**Viewer 3D của Nerfstudio:** để đổi góc nhìn trực tiếp, mang `config.yml`, checkpoint và PandaSet đã duyệt sang môi trường Linux GPU. README có lệnh `colab_task.py viewer` gọi Viser của `neurad-studio` với các đường dẫn đã chép. Đối chiếu RGB ở camera gốc bằng các PNG của Task 8.

In [ ]:
OBSERVATIONS = {
    "walking": {},
    "cycling": {},
    "occlusion": {},
}
if PHASE == "train" and TRAIN_STAGE != "smoke":
    for case in cases:
        case.setdefault("observations", {}).update(OBSERVATIONS.get(case["case"], {}))
    write_json(cases_path, cases)
    pack_demo(cases, RUNS, GPU_ROOT, PROJECT_DIR / "viewer", DEMO)
    print(f"Đã xuất demo tại {DEMO.with_suffix('.zip')}")